In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
from pyspark.sql.functions import (
    col,
    count,
    countDistinct,
    sum as spark_sum,
    avg,
    round as spark_round,
    to_date,
    concat_ws,
    coalesce,
    lit
)

adls_options = get_adls_options()

SILVER_TABLE = "physical_vendas_caixa"
SILVER_PATH = f"{SILVER_BASE_PATH}{SILVER_TABLE}"

df_vendas = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(SILVER_PATH)
    .withColumn("data_venda", to_date(col("dt_venda")))
    .withColumn("loja_caixa", concat_ws("_", col("id_loja"), col("id_caixa")))
)

print("Silver lida com sucesso.")
print(f"Total de registros: {df_vendas.count()}")

df_volume_mensal = (
    df_vendas
    .groupBy("ano", "mes")
    .agg(
        count("id_transacao").alias("qtd_transacoes"),
        spark_sum("valor_total_venda").alias("receita_total"),
        countDistinct("id_loja").alias("qtd_lojas"),
        countDistinct("id_caixa").alias("qtd_ids_caixa"),
        countDistinct("loja_caixa").alias("qtd_caixas_por_loja")
    )
    .orderBy("ano", "mes")
)

display(df_volume_mensal)

In [0]:
df_out_nov = (
    df_vendas
    .filter(
        (col("ano") == 2025) &
        (col("mes").isin(10, 11))
    )
)

df_resumo_out_nov = (
    df_out_nov
    .groupBy("ano", "mes")
    .agg(
        count("id_transacao").alias("qtd_transacoes"),
        spark_sum("valor_total_venda").alias("receita_total"),
        countDistinct("data_venda").alias("qtd_dias_com_venda"),
        countDistinct("id_loja").alias("qtd_lojas"),
        countDistinct("id_caixa").alias("qtd_ids_caixa"),
        countDistinct("loja_caixa").alias("qtd_caixas_por_loja"),
        spark_round(avg("valor_total_venda"), 2).alias("ticket_medio")
    )
    .orderBy("ano", "mes")
)

display(df_resumo_out_nov)

df_volume_diario_out_nov = (
    df_out_nov
    .groupBy("data_venda")
    .agg(
        count("id_transacao").alias("qtd_transacoes"),
        spark_sum("valor_total_venda").alias("receita_total"),
        countDistinct("id_loja").alias("qtd_lojas"),
        countDistinct("loja_caixa").alias("qtd_caixas_por_loja")
    )
    .orderBy("data_venda")
)

display(df_volume_diario_out_nov)

In [0]:
import builtins

resumo = df_resumo_out_nov.collect()

outubro = [r for r in resumo if r["mes"] == 10][0]
novembro = [r for r in resumo if r["mes"] == 11][0]

queda_abs = novembro["qtd_transacoes"] - outubro["qtd_transacoes"]

queda_pct = builtins.round(
    (queda_abs / outubro["qtd_transacoes"]) * 100,
    2
)

print("Resumo da investigação:")
print(f"Outubro/2025: {outubro['qtd_transacoes']} transações")
print(f"Novembro/2025: {novembro['qtd_transacoes']} transações")
print(f"Diferença: {queda_abs} transações")
print(f"Queda percentual: {queda_pct}%")

print("")
print(f"Dias com venda em outubro: {outubro['qtd_dias_com_venda']}")
print(f"Dias com venda em novembro: {novembro['qtd_dias_com_venda']}")

print("")
print(f"Lojas em outubro: {outubro['qtd_lojas']}")
print(f"Lojas em novembro: {novembro['qtd_lojas']}")

print("")
print(f"IDs de caixa em outubro: {outubro['qtd_ids_caixa']}")
print(f"IDs de caixa em novembro: {novembro['qtd_ids_caixa']}")

print("")
print(f"Caixas por loja em outubro: {outubro['qtd_caixas_por_loja']}")
print(f"Caixas por loja em novembro: {novembro['qtd_caixas_por_loja']}")

print("")
if novembro["qtd_dias_com_venda"] < 30:
    print("Diagnóstico provável: novembro pode estar com carga incompleta ou dias ausentes.")
elif novembro["qtd_lojas"] < outubro["qtd_lojas"]:
    print("Diagnóstico provável: houve redução de lojas com venda em novembro.")
elif novembro["qtd_caixas_por_loja"] < outubro["qtd_caixas_por_loja"]:
    print("Diagnóstico provável: houve redução de caixas por loja em novembro.")
else:
    print("Diagnóstico provável: a queda não parece ser causada por perda de lojas, caixas ou dias. Parece mudança real/recorte da base a partir de novembro/2025.")

In [0]:
df_loja_mes = (
    df_out_nov
    .groupBy("id_loja", "mes")
    .agg(
        count("id_transacao").alias("qtd_transacoes"),
        spark_sum("valor_total_venda").alias("receita_total"),
        countDistinct("loja_caixa").alias("qtd_caixas_por_loja")
    )
)

df_loja_comparativo = (
    df_loja_mes
    .groupBy("id_loja")
    .pivot("mes", [10, 11])
    .agg(
        spark_sum("qtd_transacoes")
    )
    .withColumnRenamed("10", "transacoes_outubro")
    .withColumnRenamed("11", "transacoes_novembro")
    .withColumn("transacoes_outubro", coalesce(col("transacoes_outubro"), lit(0)))
    .withColumn("transacoes_novembro", coalesce(col("transacoes_novembro"), lit(0)))
    .withColumn(
        "diferenca_transacoes",
        col("transacoes_novembro") - col("transacoes_outubro")
    )
    .withColumn(
        "queda_pct",
        spark_round(
            (col("diferenca_transacoes") / col("transacoes_outubro")) * 100,
            2
        )
    )
    .orderBy(col("diferenca_transacoes").asc())
)

display(df_loja_comparativo)